<a href="https://colab.research.google.com/github/MarcinMarud/flyrank_internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcinMarud/flyrank_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
dim_content_path = f"{rel}/dim_content.parquet"
fact_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

base = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_90d,
        SUM(f.gsc_clicks) AS clicks_90d,
        AVG(f.gsc_avg_position) AS avg_position,
        c.word_count,
        c.content_created_date,
        c.content_updated_date
    FROM read_parquet('{fact_path}') f
    JOIN read_parquet('{dim_content_path}') c USING (content_hash_id)
    GROUP BY 1, c.word_count, c.content_created_date, c.content_updated_date
""").df()

base["ctr"] = base["clicks_90d"] / base["impressions_90d"].replace(0, pd.NA)
base["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(base["content_updated_date"])).dt.days

base["staleness_bucket"] = pd.cut(
    base["days_since_update"],
    bins=[-1, 90, 180, 365, 10_000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
staleness_table = base.groupby("staleness_bucket").agg(
    n=("content_hash_id", "count"),
    median_impressions=("impressions_90d", "median"),
    median_ctr=("ctr", "median")
).reset_index()

base["position_bucket"] = pd.cut(
    base["avg_position"],
    bins=[0, 3, 10, 20, 100],
    labels=["1-3", "4-10", "11-20", "21+"]
)
position_table = base.groupby("position_bucket").agg(
    n=("content_hash_id", "count"),
    median_ctr=("ctr", "median"),
    median_impressions=("impressions_90d", "median")
).reset_index()

print(staleness_table)
print(position_table)

mask = (base["days_since_update"] >= 180) & (base["impressions_90d"] >= 500)
base["reason_code"] = None
base.loc[mask, "reason_code"] = "stale_visible_page"
base["action"] = "monitor"
base.loc[mask, "action"] = "review_for_refresh"
base["baseline_score"] = 0
base.loc[mask, "baseline_score"] = (
    base.loc[mask, "days_since_update"] / base["days_since_update"].max()
    + base.loc[mask, "impressions_90d"] / base["impressions_90d"].max()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_bucket      n  median_impressions median_ctr
0             <90d  30655               196.0        0.0
1          90-180d   3608                 0.0        0.0
2         180-365d   3816                 0.0        0.0
3            365d+      0                 NaN        NaN
  position_bucket      n median_ctr  median_impressions
0             1-3  16144        0.0               249.0
1            4-10  81988        0.0               201.0
2           11-20  32203        0.0               257.0
3             21+  44867        0.0               119.0


/tmp/ipykernel_546/2721138254.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = base.groupby("staleness_bucket").agg(
/tmp/ipykernel_546/2721138254.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  position_table = base.groupby("position_bucket").agg(
/tmp/ipykernel_546/2721138254.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.8784348  0.77162351 0.76326747 0.76333553 0.76356563 0.8174971 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  base.loc[mask, "

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring and ranking every page

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue = base[base["reason_code"].notna()].sort_values("baseline_score", ascending=False)
print(len(queue))

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
queue.head()

6


,content_hash_id,impressions_90d,clicks_90d,avg_position,word_count,content_created_date,content_updated_date,ctr,days_since_update,staleness_bucket,position_bucket,reason_code,action,baseline_score
9793,content_42ce26be1ec6be00,4411.0,6.0,4.262553,1421,2025-07-10,2025-07-10,0.00136,264,180-365d,4-10,stale_visible_page,review_for_refresh,0.878435
175413,content_5120dcbbb086843d,1429.0,0.0,6.321173,<NA>,2025-07-27,2025-07-27,0.0,247,180-365d,4-10,stale_visible_page,review_for_refresh,0.817497
38159,content_bea86ce3455100b0,3670.0,1.0,6.555793,1447,2025-08-11,2025-08-11,0.000272,232,180-365d,4-10,stale_visible_page,review_for_refresh,0.771624
174242,content_eba53d72e18a9f93,734.0,2.0,5.234187,1504,2025-08-12,2025-08-12,0.002725,231,180-365d,4-10,stale_visible_page,review_for_refresh,0.763566
174239,content_c126a43258b574c3,592.0,0.0,26.359739,1860,2025-08-12,2025-08-12,0.0,231,180-365d,21+,stale_visible_page,review_for_refresh,0.763336


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
for i, row in top20.iterrows():
    print(row["content_hash_id"][:8], row["action"], row["reason_code"], round(row["baseline_score"], 2), row["impressions_90d"], row["days_since_update"])

content_ review_for_refresh stale_visible_page 0.88 4411.0 264
content_ review_for_refresh stale_visible_page 0.82 1429.0 247
content_ review_for_refresh stale_visible_page 0.77 3670.0 232
content_ review_for_refresh stale_visible_page 0.76 734.0 231
content_ review_for_refresh stale_visible_page 0.76 592.0 231
content_ review_for_refresh stale_visible_page 0.76 550.0 231


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak picks: rows near the threshold cutoff - likely noise rather than real signal

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(base["days_since_update"].describe())
print(base["impressions_90d"].describe())
print(queue.tail(10))

count    331437.000000
mean        -44.484029
std          48.485626
min         -97.000000
25%         -62.000000
50%         -50.000000
75%         -50.000000
max         303.000000
Name: days_since_update, dtype: float64
count    331437.000000
mean        846.790156
std        4044.514753
min           0.000000
25%           0.000000
50%           2.000000
75%         216.000000
max      617124.000000
Name: impressions_90d, dtype: float64
                 content_hash_id  impressions_90d  clicks_90d  avg_position  \
9793    content_42ce26be1ec6be00           4411.0         6.0      4.262553   
175413  content_5120dcbbb086843d           1429.0         0.0      6.321173   
38159   content_bea86ce3455100b0           3670.0         1.0      6.555793   
174242  content_eba53d72e18a9f93            734.0         2.0      5.234187   
174239  content_c126a43258b574c3            592.0         0.0     26.359739   
38161   content_5271624ae98fff86            550.0         3.0      6.532108   



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.